In [ ]:
import duckdb
import os
import pandas as pd
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats


In [ ]:
unique_ids_file = "../unique_ids_final.csv"
did_input = "did_input_final.parquet"

In [ ]:
ROOT_DIR = Path.cwd()
REPO_ROOT = ROOT_DIR.parent
sys.path.insert(0, str(REPO_ROOT))
from data_preparation.config import connect_db

# Paths
DB_NAME = REPO_ROOT / "insurance.duckdb"
unique_ids_file = REPO_ROOT / "unique_ids_final.csv"
did_input = REPO_ROOT / "did_input_final.parquet"

# Connect to DuckDB using shared configuration
con = connect_db(str(DB_NAME))

# Analysis-specific DuckDB settings
TEMP_DIR = ROOT_DIR / "tmp" / "duckdb"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

con.execute("SET memory_limit = '48GB'")
con.execute("SET threads = 8")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET max_temp_directory_size = '14GiB'")
con.execute("SET preserve_insertion_order = false")

print(con.execute(
    "SELECT current_setting('memory_limit'), current_setting('threads'), current_setting('temp_directory')"
).fetchall())

# --- final analysed IDs + DiD input ---
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE final_unique_ids AS
    SELECT * FROM read_csv_auto('{unique_ids_file.as_posix()}')
""")
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW did_input_final AS
    SELECT * FROM read_parquet('{did_input.as_posix()}')
""")

print("final_unique_ids rows:", con.execute("SELECT COUNT(*) FROM final_unique_ids").fetchone()[0])
print("did_input_final rows:", con.execute("SELECT COUNT(*) FROM did_input_final").fetchone()[0])
print(con.execute("DESCRIBE final_unique_ids").fetchdf())
print(con.execute("DESCRIBE did_input_final").fetchdf())

In [ ]:
# --- adoption month (start_month equivalent) + constant covariates -------
con.execute("""
    CREATE OR REPLACE TABLE final_start_month AS
    SELECT
        eth_id,
        t_period_start AS start_month
    FROM did_input_final
    WHERE event_time = 0
""")
print("final_start_month rows:", con.execute("SELECT COUNT(*) FROM final_start_month").fetchone()[0])
print("distinct eth_ids:", con.execute("SELECT COUNT(DISTINCT eth_id) FROM final_start_month").fetchone()[0])

# age/gender AT ADOPTION -> use event_time = 0, not DISTINCT over all periods
con.execute("""
    CREATE OR REPLACE TABLE final_demo AS
    SELECT eth_id, age, gender
    FROM did_input_final
    WHERE event_time = 0
""")
n_rows = con.execute("SELECT COUNT(*) FROM final_demo").fetchone()[0]
n_ids  = con.execute("SELECT COUNT(DISTINCT eth_id) FROM final_demo").fetchone()[0]
print(f"final_demo rows={n_rows}, distinct eth_ids={n_ids}")

final_panel = con.execute("""
    SELECT
        u.eth_id,
        sm.start_month,
        d.age,
        d.gender,
        inc.einkommensindex AS income_index
    FROM final_unique_ids u
    LEFT JOIN final_start_month sm USING (eth_id)
    LEFT JOIN final_demo d USING (eth_id)
    LEFT JOIN (
        SELECT eth_id, einkommensindex,
               ROW_NUMBER() OVER (PARTITION BY eth_id ORDER BY endedatum DESC) AS rn
        FROM income_index
    ) inc
        ON inc.eth_id = u.eth_id AND inc.rn = 1
""").fetchdf()

print("rows:", len(final_panel))
print("missing start_month:", final_panel['start_month'].isna().sum())
print("missing income_index:", final_panel['income_index'].isna().sum())

final_panel.head()

## 1. Is it more people, or the same people filing more?

In [ ]:
margin_check = con.execute("""
    WITH windowed AS (
        SELECT
            o.eth_id,
            o.betrag,
            CASE
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 3 MONTH
                     AND o.behandlungsbeginn <  sm.start_month
                THEN 'pre_1_3'
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 6 MONTH
                     AND o.behandlungsbeginn <  sm.start_month - INTERVAL 3 MONTH
                THEN 'pre_4_6'
            END AS period_window
        FROM outpatient o
        INNER JOIN final_unique_ids u ON u.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
    )
    SELECT
        period_window,
        COUNT(DISTINCT eth_id)                      AS n_claimants,
        COUNT(*)                                    AS n_claims,
        COUNT(*) * 1.0 / COUNT(DISTINCT eth_id)      AS claims_per_claimant,
        SUM(betrag)                                 AS total_cost,
        SUM(betrag) / COUNT(*)                       AS cost_per_claim,
        SUM(betrag) / COUNT(DISTINCT eth_id)          AS cost_per_claimant
    FROM windowed
    WHERE period_window IS NOT NULL
    GROUP BY period_window
    ORDER BY period_window DESC
""").fetchdf()

margin_check

In [ ]:
new_vs_returning = con.execute("""
    WITH claimant_history AS (
        SELECT
            o.eth_id,
            sm.start_month,
            MIN(o.behandlungsbeginn) FILTER (
                WHERE o.behandlungsbeginn >= sm.start_month - INTERVAL 12 MONTH
                  AND o.behandlungsbeginn <  sm.start_month - INTERVAL 6 MONTH
            ) AS had_claim_months_7_12
        FROM outpatient o
        INNER JOIN final_unique_ids u ON u.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
        GROUP BY o.eth_id, sm.start_month
    ),
    pre_1_3_claimants AS (
        SELECT DISTINCT o.eth_id
        FROM outpatient o
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
          AND o.behandlungsbeginn >= sm.start_month - INTERVAL 3 MONTH
          AND o.behandlungsbeginn <  sm.start_month
    )
    SELECT
        CASE WHEN ch.had_claim_months_7_12 IS NULL THEN 'no_prior_claims_7_12mo' ELSE 'had_prior_claims_7_12mo' END AS claimant_type,
        COUNT(*) AS n_claimants
    FROM pre_1_3_claimants p
    LEFT JOIN claimant_history ch ON ch.eth_id = p.eth_id
    GROUP BY 1
""").fetchdf()

new_vs_returning

## 2. Returning claimants - shift in care?

In [ ]:
# Returning claimants = had outpatient activity in months -7 to -12
returning_claimants = con.execute("""
    WITH claimant_history AS (
        SELECT
            o.eth_id,
            sm.start_month,
            MIN(o.behandlungsbeginn) FILTER (
                WHERE o.behandlungsbeginn >= sm.start_month - INTERVAL 12 MONTH
                  AND o.behandlungsbeginn <  sm.start_month - INTERVAL 6 MONTH
            ) AS had_claim_months_7_12
        FROM outpatient o
        INNER JOIN final_unique_ids u ON u.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
        GROUP BY o.eth_id, sm.start_month
    )
    SELECT eth_id, start_month
    FROM claimant_history
    WHERE had_claim_months_7_12 IS NOT NULL
""").fetchdf()

con.execute("CREATE OR REPLACE TABLE returning_claimants AS SELECT * FROM returning_claimants")
print("returning claimants:", len(returning_claimants))

In [ ]:
# --- behandlungsgrund composition, returning claimants only --------------
grund_breakdown = con.execute("""
    WITH windowed AS (
        SELECT
            o.eth_id,
            o.betrag,
            TRIM(o.behandlungsgrund) AS behandlungsgrund,
            CASE
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 3 MONTH
                     AND o.behandlungsbeginn <  sm.start_month
                THEN 'pre_1_3'
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 6 MONTH
                     AND o.behandlungsbeginn <  sm.start_month - INTERVAL 3 MONTH
                THEN 'pre_4_6'
            END AS period_window
        FROM outpatient o
        INNER JOIN returning_claimants r ON r.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
    )
    SELECT
        period_window,
        behandlungsgrund,
        COUNT(*)                                AS n_claims,
        COUNT(DISTINCT eth_id)                  AS n_claimants,
        SUM(betrag)                             AS total_cost,
        SUM(betrag) / COUNT(*)                  AS cost_per_claim
    FROM windowed
    WHERE period_window IS NOT NULL
    GROUP BY period_window, behandlungsgrund
    ORDER BY period_window DESC, total_cost DESC
""").fetchdf()

grund_breakdown

In [ ]:
# --- produktkategorie composition, returning claimants only --------------
produkt_breakdown = con.execute("""
    WITH windowed AS (
        SELECT
            o.eth_id,
            o.betrag,
            TRIM(o.produktkategorie) AS produktkategorie,
            CASE
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 3 MONTH
                     AND o.behandlungsbeginn <  sm.start_month
                THEN 'pre_1_3'
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 6 MONTH
                     AND o.behandlungsbeginn <  sm.start_month - INTERVAL 3 MONTH
                THEN 'pre_4_6'
            END AS period_window
        FROM outpatient o
        INNER JOIN returning_claimants r ON r.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
    )
    SELECT
        period_window,
        produktkategorie,
        COUNT(*)                                AS n_claims,
        SUM(betrag)                             AS total_cost,
        SUM(betrag) / COUNT(*)                  AS cost_per_claim
    FROM windowed
    WHERE period_window IS NOT NULL
    GROUP BY period_window, produktkategorie
    ORDER BY period_window DESC, total_cost DESC
""").fetchdf()

produkt_breakdown

In [ ]:
# --- tarifposition, top movers only (there may be many categories) -------
tarif_breakdown = con.execute("""
    WITH windowed AS (
        SELECT
            o.eth_id,
            o.betrag,
            TRIM(o.tarifposition) AS tarifposition,
            CASE
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 3 MONTH
                     AND o.behandlungsbeginn <  sm.start_month
                THEN 'pre_1_3'
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 6 MONTH
                     AND o.behandlungsbeginn <  sm.start_month - INTERVAL 3 MONTH
                THEN 'pre_4_6'
            END AS period_window
        FROM outpatient o
        INNER JOIN returning_claimants r ON r.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
    )
    SELECT
        tarifposition,
        SUM(total_cost) FILTER (WHERE period_window = 'pre_1_3') AS cost_pre_1_3,
        SUM(total_cost) FILTER (WHERE period_window = 'pre_4_6') AS cost_pre_4_6,
        SUM(total_cost) FILTER (WHERE period_window = 'pre_1_3')
            - SUM(total_cost) FILTER (WHERE period_window = 'pre_4_6') AS cost_delta
    FROM (
        SELECT tarifposition, period_window, eth_id, SUM(betrag) AS total_cost
        FROM windowed
        WHERE period_window IS NOT NULL
        GROUP BY tarifposition, period_window, eth_id
    )
    GROUP BY tarifposition
    ORDER BY cost_delta DESC
    LIMIT 20
""").fetchdf()

tarif_breakdown

01.03.1000 is Sars-CoV-2-Selbsttest (Abgabe direkt in Apotheke)

In [ ]:
tarif_system_delta = con.execute("""
    WITH windowed AS (
        SELECT
            o.eth_id,
            o.betrag,
            TRIM(o.tarif) AS tarif,
            TRIM(o.tarifposition) AS tarifposition,
            CASE
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 3 MONTH
                     AND o.behandlungsbeginn <  sm.start_month
                THEN 'pre_1_3'
                WHEN o.behandlungsbeginn >= sm.start_month - INTERVAL 6 MONTH
                     AND o.behandlungsbeginn <  sm.start_month - INTERVAL 3 MONTH
                THEN 'pre_4_6'
            END AS period_window
        FROM outpatient o
        INNER JOIN returning_claimants r ON r.eth_id = o.eth_id
        INNER JOIN final_start_month sm ON sm.eth_id = o.eth_id
        WHERE TRIM(o.gesetz) = 'KVG'
          AND TRIM(o.tarifposition) IN (
              '01.03.1000','7301','PA010','01.01.1150','02.0020','7786293',
              '7311','3888948','7693791','00.0020','99.9022.01.31','00.0010',
              '4756580','C10','41010','7805111','7744532','4648471','7782364','999.5000'
          )
    )
    SELECT
        tarif,
        tarifposition,
        SUM(betrag) FILTER (WHERE period_window = 'pre_1_3') AS cost_pre_1_3,
        SUM(betrag) FILTER (WHERE period_window = 'pre_4_6') AS cost_pre_4_6,
        SUM(betrag) FILTER (WHERE period_window = 'pre_1_3')
            - SUM(betrag) FILTER (WHERE period_window = 'pre_4_6') AS cost_delta,
        COUNT(*) FILTER (WHERE period_window = 'pre_1_3') AS n_claims_pre_1_3,
        COUNT(*) FILTER (WHERE period_window = 'pre_4_6') AS n_claims_pre_4_6
    FROM windowed
    WHERE period_window IS NOT NULL
    GROUP BY tarif, tarifposition
    ORDER BY cost_delta DESC
""").fetchdf()

tarif_system_delta

Cost delta:
1. 01.03.1000: Sars-CoV-2-Selbsttest (Abgabe direkt in Apotheke)
2. 7301: Einzelsitzungspauschale für allgemeine Physiotherapie
3. PA010: Psychotherapie Tarif,"Diagnostik und Therapie mit einem Patienten in Anwesenheit, pro 1 Min."

## 3. When were these COVID-tests done?

In [ ]:
# Self-test billing by CALENDAR month
covid_test_calendar = con.execute("""
    SELECT
        DATE_TRUNC('month', o.behandlungsbeginn) AS calendar_month,
        COUNT(*) AS n_claims,
        SUM(o.betrag) AS total_cost
    FROM outpatient o
    INNER JOIN returning_claimants r ON r.eth_id = o.eth_id
    WHERE TRIM(o.gesetz) = 'KVG'
      AND TRIM(o.tarifposition) = '01.03.1000'
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

covid_test_calendar

In [ ]:
adopter_calendar = con.execute("""
    SELECT DATE_TRUNC('month', start_month) AS calendar_month, COUNT(*) AS n_adopters
    FROM final_start_month
    GROUP BY 1 ORDER BY 1
""").fetchdf()

adopter_calendar

In [ ]:
print(covid_test_calendar.columns.tolist())

In [ ]:
def plot_adopters_vs_selftests_quarterly(adopter_calendar, covid_test_calendar):
    df_a = adopter_calendar.copy()
    df_c = covid_test_calendar.copy()

    # auto-detect the non-date column names
    a_val_col = [c for c in df_a.columns if c != 'calendar_month'][0]
    c_val_col = [c for c in df_c.columns if c != 'calendar_month'][0]

    df_a['calendar_month'] = pd.to_datetime(df_a['calendar_month'])
    df_c['calendar_month'] = pd.to_datetime(df_c['calendar_month'])

    df_a['quarter'] = df_a['calendar_month'].dt.to_period('Q').dt.to_timestamp()
    df_c['quarter'] = df_c['calendar_month'].dt.to_period('Q').dt.to_timestamp()

    df_a = df_a.groupby('quarter', as_index=False)[a_val_col].sum()
    df_c = df_c.groupby('quarter', as_index=False)[c_val_col].sum()

    full_range = pd.date_range(
        start=min(df_a['quarter'].min(), df_c['quarter'].min()),
        end=max(df_a['quarter'].max(), df_c['quarter'].max()),
        freq='QS'
    )
    full_df = pd.DataFrame({'quarter': full_range})
    full_df = full_df.merge(df_a, on='quarter', how='left').fillna({a_val_col: 0})
    full_df = full_df.merge(df_c, on='quarter', how='left').fillna({c_val_col: 0})

    print(f"Plotting {len(full_df)} quarters from {full_df['quarter'].min().date()} "
          f"to {full_df['quarter'].max().date()}")
    print(f"Using columns: adopters={a_val_col!r}, self-tests={c_val_col!r}")

    fig, ax1 = plt.subplots(figsize=(12, 6))
    color1 = '#1f77b4'
    ax1.set_xlabel('Quarter')
    ax1.set_ylabel('Number of adopters', color=color1)
    ax1.plot(full_df['quarter'], full_df[a_val_col],
              color=color1, marker='o', markersize=4, label='Adopters')
    ax1.tick_params(axis='y', labelcolor=color1)

    ax2 = ax1.twinx()
    color2 = '#d62728'
    ax2.set_ylabel('Number of self-test claims (01.03.1000)', color=color2)
    ax2.plot(full_df['quarter'], full_df[c_val_col],
              color=color2, marker='o', markersize=4, label='Self-tests')
    ax2.tick_params(axis='y', labelcolor=color2)

    for date_str, label in [('2021-03-15', 'Free testing for all'), ('2021-04-07', 'Self-tests in pharmacies')]:
        ax1.axvline(pd.Timestamp(date_str), color='gray', linestyle='--', alpha=0.5)
        ax1.text(pd.Timestamp(date_str), ax1.get_ylim()[1]*0.95, label,
                  rotation=90, fontsize=8, va='top', ha='right', color='gray')

    fig.suptitle('Adopters vs. COVID self-test claims by quarter')
    fig.tight_layout()
    plt.savefig('./adopters_vs_selftests_quarterly.png', dpi=150, bbox_inches='tight')
    plt.show()

    return full_df

full_df = plot_adopters_vs_selftests_quarterly(adopter_calendar, covid_test_calendar)

In [ ]:
# Check actual granularity of start_month values
print(sorted(con.execute("SELECT DISTINCT start_month FROM final_start_month ORDER BY start_month").fetchdf()['start_month'].astype(str).tolist()))

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE first_adoption_month AS
    SELECT
        CSS_MARKETING_ID AS eth_id,
        DATE_TRUNC('month', MIN(CAST(TRANSACTION_DATETIME AS DATE))) AS first_month
    FROM a365_transaction_check
    WHERE TRANSACTION_DATETIME IS NOT NULL
      AND CSS_MARKETING_ID IN (SELECT eth_id FROM final_unique_ids)
    GROUP BY CSS_MARKETING_ID
""")

first_adoption_df = con.execute("SELECT * FROM first_adoption_month ORDER BY first_month").fetchdf()
first_adoption_df.head(20)

print("rows:", con.execute("SELECT COUNT(*) FROM first_adoption_month").fetchone()[0])
print("distinct eth_ids:", con.execute("SELECT COUNT(DISTINCT eth_id) FROM first_adoption_month").fetchone()[0])

In [ ]:
first_adoption_df.head(20)

## 4. How do adopters differ from later adopters in outpatient care?

In [ ]:
# --- Build treated and control person-quarter groups across ALL cohorts ---

con.execute("""
    CREATE OR REPLACE TABLE cohort_groups AS
    WITH panel AS (
        SELECT eth_id, t_index, g_index, t_period_start
        FROM did_input_final
    ),
    cohorts AS (
        SELECT DISTINCT g_index AS g FROM panel WHERE g_index IS NOT NULL
    ),
    treated AS (
        SELECT p.eth_id, p.g_index AS g, p.t_period_start AS quarter_start, 'treated' AS role
        FROM panel p
        WHERE p.g_index IS NOT NULL AND p.t_index = p.g_index - 1
    ),
    control AS (
        SELECT p.eth_id, c.g AS g, p.t_period_start AS quarter_start, 'control' AS role
        FROM panel p
        JOIN cohorts c ON p.t_index = c.g - 1
        WHERE p.g_index IS NULL OR p.g_index > c.g
    )
    SELECT * FROM treated
    UNION ALL
    SELECT * FROM control
""")

print(con.execute("SELECT role, COUNT(*) AS n_person_periods, COUNT(DISTINCT eth_id) AS n_unique_people FROM cohort_groups GROUP BY role").fetchdf())

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE cohort_claims AS
    SELECT
        cg.eth_id,
        cg.g,
        cg.role,
        o.behandlungsbeginn,
        TRIM(o.behandlungsgrund) AS behandlungsgrund,
        TRIM(o.tarifposition)    AS tarifposition,
        TRIM(o.produktkategorie) AS produktkategorie,
        o.betrag
    FROM cohort_groups cg
    JOIN outpatient o ON o.eth_id = cg.eth_id
    WHERE TRIM(o.gesetz) = 'KVG'
      AND o.behandlungsbeginn >= cg.quarter_start
      AND o.behandlungsbeginn <  cg.quarter_start + INTERVAL 3 MONTH
""")

print("Total claims pulled:", con.execute("SELECT COUNT(*) FROM cohort_claims").fetchone()[0])
print(con.execute("SELECT role, COUNT(*) AS n_claims, COUNT(DISTINCT eth_id) AS n_claimants FROM cohort_claims GROUP BY role").fetchdf())

In [ ]:
person_level = con.execute("""
    SELECT
        cg.eth_id,
        cg.g,
        cg.role,
        COUNT(cc.behandlungsbeginn)      AS n_claims,
        COALESCE(SUM(cc.betrag), 0)      AS total_cost
    FROM cohort_groups cg
    LEFT JOIN cohort_claims cc
        ON cc.eth_id = cg.eth_id AND cc.g = cg.g AND cc.role = cg.role
    GROUP BY cg.eth_id, cg.g, cg.role
""").fetchdf()

person_level.head(100)

In [ ]:
treated_n = person_level.loc[person_level['role']=='treated', 'n_claims']
control_n = person_level.loc[person_level['role']=='control', 'n_claims']
treated_cost = person_level.loc[person_level['role']=='treated', 'total_cost']
control_cost = person_level.loc[person_level['role']=='control', 'total_cost']

u_claims, p_claims = stats.mannwhitneyu(treated_n, control_n, alternative='two-sided')
u_cost, p_cost = stats.mannwhitneyu(treated_cost, control_cost, alternative='two-sided')

print("=== Person-level tests ===")
print(f"n_claims/person  -> treated mean={treated_n.mean():.4f}, control mean={control_n.mean():.4f}, Mann-Whitney p={p_claims:.2e}")
print(f"total_cost/person -> treated mean={treated_cost.mean():.2f}, control mean={control_cost.mean():.2f}, Mann-Whitney p={p_cost:.2e}")

# --- behandlungsgrund composition ---
claims_df = con.execute("SELECT role, behandlungsgrund FROM cohort_claims").fetchdf()
ct_grund = pd.crosstab(claims_df['behandlungsgrund'], claims_df['role'])
chi2, p_grund, dof, expected = stats.chi2_contingency(ct_grund)
print("\n=== behandlungsgrund composition ===")
print(ct_grund)
print(f"Chi-square p={p_grund:.2e}")

# proportions for readability
print((ct_grund / ct_grund.sum(axis=0)).round(4))

# --- top tarifposition codes (restrict to codes with meaningful volume to avoid sparse-cell chi-square issues) ---
tarif_counts = con.execute("""
    SELECT tarifposition, role, COUNT(*) AS n
    FROM cohort_claims
    GROUP BY tarifposition, role
""").fetchdf()

top_codes = (tarif_counts.groupby('tarifposition')['n'].sum()
             .sort_values(ascending=False).head(20).index)

ct_tarif = tarif_counts[tarif_counts['tarifposition'].isin(top_codes)].pivot_table(
    index='tarifposition', columns='role', values='n', fill_value=0
)
chi2_t, p_tarif, dof_t, expected_t = stats.chi2_contingency(ct_tarif)
print("\n=== top-20 tarifposition composition ===")
print(ct_tarif.sort_values('treated', ascending=False))
print(f"Chi-square p={p_tarif:.2e}")

In [ ]:
# Rebuild cohort_claims to include tarif
con.execute("""
    CREATE OR REPLACE TABLE cohort_claims AS
    SELECT
        cg.eth_id,
        cg.g,
        cg.role,
        o.behandlungsbeginn,
        TRIM(o.tarif)            AS tarif,
        TRIM(o.behandlungsgrund) AS behandlungsgrund,
        TRIM(o.tarifposition)    AS tarifposition,
        TRIM(o.produktkategorie) AS produktkategorie,
        o.betrag
    FROM cohort_groups cg
    JOIN outpatient o ON o.eth_id = cg.eth_id
    WHERE TRIM(o.gesetz) = 'KVG'
      AND o.behandlungsbeginn >= cg.quarter_start
      AND o.behandlungsbeginn <  cg.quarter_start + INTERVAL 3 MONTH
""")

print("Total claims pulled:", con.execute("SELECT COUNT(*) FROM cohort_claims").fetchone()[0])

In [ ]:
tarif_props = con.execute("""
    WITH totals AS (
        SELECT role, COUNT(*) AS total_claims FROM cohort_claims GROUP BY role
    )
    SELECT
        cc.tarif,
        cc.tarifposition,
        SUM(CASE WHEN cc.role='treated' THEN 1 ELSE 0 END) AS n_treated,
        SUM(CASE WHEN cc.role='control' THEN 1 ELSE 0 END) AS n_control,
        SUM(CASE WHEN cc.role='treated' THEN 1 ELSE 0 END) * 1.0 /
            MAX(CASE WHEN cc.role='treated' THEN t.total_claims END) AS pct_of_treated,
        SUM(CASE WHEN cc.role='control' THEN 1 ELSE 0 END) * 1.0 /
            MAX(CASE WHEN cc.role='control' THEN t.total_claims END) AS pct_of_control
    FROM cohort_claims cc
    CROSS JOIN totals t
    WHERE t.role = cc.role
    GROUP BY cc.tarif, cc.tarifposition
    HAVING n_treated + n_control > 5000
    ORDER BY (SUM(CASE WHEN cc.role='treated' THEN 1 ELSE 0 END) * 1.0 /
               MAX(CASE WHEN cc.role='treated' THEN t.total_claims END))
           - (SUM(CASE WHEN cc.role='control' THEN 1 ELSE 0 END) * 1.0 /
               MAX(CASE WHEN cc.role='control' THEN t.total_claims END)) DESC
""").fetchdf()

tarif_props

In [ ]:
pd.set_option('display.max_rows', 50)

print("=== TOP 15 — overrepresented in treated ===")
print(tarif_props.sort_values('pct_of_treated', ascending=False).eval('diff = pct_of_treated - pct_of_control').nlargest(15, 'diff')[['tarif','tarifposition','n_treated','n_control','pct_of_treated','pct_of_control','diff']])

print("\n=== BOTTOM 15 — underrepresented in treated ===")
print(tarif_props.assign(diff=tarif_props['pct_of_treated'] - tarif_props['pct_of_control']).nsmallest(15, 'diff')[['tarif','tarifposition','n_treated','n_control','pct_of_treated','pct_of_control','diff']])

In [ ]:
con.close()